# Sign → Text — Complete Pipeline (Extraction → 3 Models → Results)

End-to-end and reproducible: extracts landmark features from your videos, trains
**three architectures**, and produces every evaluation artifact for the thesis.

**Feature extraction uses MediaPipe VIDEO mode with a fresh landmarker instance per
video** (frames processed in order with temporal tracking, then 30 sampled). This is
identical to `app/core/sign_features.py`, so training and the live app share one
preprocessing path. Data layout: `recordings/<GLOSS>/<clip>.mp4`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Runtime > Change runtime type > GPU  (for the RNN/Transformer training)

In [ ]:
!pip -q install mediapipe opencv-python scikit-learn pandas
!wget -q -O hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

In [ ]:
import os, json, time, numpy as np, tensorflow as tf
import matplotlib.pyplot as plt, pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             cohen_kappa_score, matthews_corrcoef, roc_auc_score, top_k_accuracy_score)

DATASET_PATH = '/content/drive/MyDrive/recordings'
TASK_PATH    = 'hand_landmarker.task'
OUT_DIR      = '/content/drive/MyDrive/HAND_MODEL'
METRICS_DIR  = os.path.join(OUT_DIR, 'thesis_metrics')
os.makedirs(METRICS_DIR, exist_ok=True)

FORCE_REEXTRACT = True     # data changed -> re-extract
AUG_PER_SAMPLE  = 12
TEST_SPLIT, VAL_SPLIT = 0.15, 0.15
EPOCHS, BATCH_SIZE, SEED = 120, 32, 42
np.random.seed(SEED); tf.random.set_seed(SEED); plt.rcParams['figure.dpi'] = 130

## Shared feature code — IDENTICAL to `app/core/sign_features.py` (VIDEO mode)

In [ ]:
"""
sign_features.py  —  SHARED feature extraction for Sign->Text.

  *** PARITY CONTRACT ***
  This exact file is used BOTH in training (Colab notebook) and in the live app.
  If you change anything here, you MUST retrain and ship the same file to the app.
  Do not "improve" one side only — that is what broke accuracy before.

Pipeline (identical on both sides):
  1. Read video / camera frames as RGB, full resolution.
  2. Run MediaPipe HandLandmarker in IMAGE mode (frame-independent -> sampling
     density doesn't matter, so train and live always agree).
  3. Order hands deterministically by handedness: slot 0 = Left, slot 1 = Right.
     Missing hand -> that 63-float block stays zero.
  4. Uniformly sample SEQ_LEN frames across the whole single-gloss clip/segment.
  5. Normalize the WHOLE sequence: subtract the clip's mean hand anchor and divide
     by mean hand size -> position- and scale-invariant, motion preserved.
"""

import numpy as np

try:
    import cv2
    import mediapipe as mp
    from mediapipe.tasks import python as mp_python
    from mediapipe.tasks.python import vision
    _MP_OK = True
except Exception:
    _MP_OK = False

SEQ_LEN = 30
HANDS = 2
LM = 21
C = 3
FEATURES = HANDS * LM * C  # 126

WRIST = 0
MIDDLE_MCP = 9


# ---------------------------------------------------------------------------
# MediaPipe landmarker (IMAGE mode = frame-independent = train/live parity)
# ---------------------------------------------------------------------------
def make_landmarker(task_path: str, mode: str = "image", result_callback=None):
    m = str(mode).lower()
    if m == "video":
        rm = vision.RunningMode.VIDEO
    elif m in ("live_stream", "live", "stream"):
        rm = vision.RunningMode.LIVE_STREAM
    else:
        rm = vision.RunningMode.IMAGE

    base = mp_python.BaseOptions(model_asset_path=task_path)
    kwargs = dict(
        base_options=base,
        running_mode=rm,
        num_hands=2,
        min_hand_detection_confidence=0.5,
        min_hand_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    if rm == vision.RunningMode.LIVE_STREAM:
        kwargs["result_callback"] = result_callback
    opts = vision.HandLandmarkerOptions(**kwargs)
    return vision.HandLandmarker.create_from_options(opts)


def detect_image(landmarker, rgb_frame):
    """IMAGE mode: synchronous, returns result."""
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    return landmarker.detect(mp_image)


def detect_video(landmarker, rgb_frame, ts_ms):
    """VIDEO mode: synchronous w/ tracking. ts_ms must strictly increase."""
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    return landmarker.detect_for_video(mp_image, int(ts_ms))


def detect_async(landmarker, rgb_frame, ts_ms):
    """LIVE_STREAM mode: non-blocking; result arrives via result_callback.
    ts_ms must strictly increase."""
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    landmarker.detect_async(mp_image, int(ts_ms))


# ---------------------------------------------------------------------------
# Result -> (kp[126], present[2])  ordered by handedness
# ---------------------------------------------------------------------------
def landmarks_from_result(result):
    kp = np.zeros(FEATURES, dtype=np.float32)
    present = np.zeros(HANDS, dtype=np.float32)
    hands = getattr(result, "hand_landmarks", None)
    if not hands:
        return kp, present

    n = min(2, len(hands))
    slot = [None, None]  # [Left, Right]
    handed = getattr(result, "handedness", None)
    for i in range(n):
        cat = None
        try:
            cat = handed[i][0].category_name
        except Exception:
            cat = None
        if cat == "Left" and slot[0] is None:
            slot[0] = i
        elif cat == "Right" and slot[1] is None:
            slot[1] = i

    used = {x for x in slot if x is not None}
    remaining = [i for i in range(n) if i not in used]
    for s in range(2):
        if slot[s] is None and remaining:
            slot[s] = remaining.pop(0)

    for s in range(2):
        i = slot[s]
        if i is None:
            continue
        present[s] = 1.0
        for j, lm in enumerate(hands[i]):
            base = s * 63 + j * 3
            kp[base:base + 3] = (lm.x, lm.y, lm.z)
    return kp, present


# ---------------------------------------------------------------------------
# Sequence normalization (position + scale invariant, motion preserved)
# ---------------------------------------------------------------------------
def normalize_sequence(seq, present):
    """seq: (T, 126) raw handedness-ordered. present: (T, 2). Returns (T, 126)."""
    seq = np.asarray(seq, dtype=np.float32).copy()
    T = seq.shape[0]
    pts = seq.reshape(T, HANDS, LM, C)
    pres = np.asarray(present, dtype=np.float32).reshape(T, HANDS) > 0.5

    anchors = []
    sizes = []
    for t in range(T):
        ws, ss = [], []
        for h in range(HANDS):
            if pres[t, h]:
                wrist = pts[t, h, WRIST]
                mcp = pts[t, h, MIDDLE_MCP]
                ws.append(wrist)
                ss.append(np.linalg.norm(mcp - wrist))
        if ws:
            anchors.append(np.mean(ws, axis=0))
            sizes.append(np.mean(ss))

    if not anchors:
        return np.zeros((T, FEATURES), dtype=np.float32)

    origin = np.mean(anchors, axis=0)
    scale = float(np.mean(sizes))
    if scale < 1e-6:
        scale = 1e-6

    out = np.zeros_like(pts)
    for t in range(T):
        for h in range(HANDS):
            if pres[t, h]:
                out[t, h] = (pts[t, h] - origin) / scale
    return out.reshape(T, FEATURES).astype(np.float32)


def sample_indices(n_frames: int, k: int = SEQ_LEN):
    return np.linspace(0, max(n_frames - 1, 0), k).astype(int)


# ---------------------------------------------------------------------------
# Whole-video -> normalized features (used in training)
# ---------------------------------------------------------------------------
def raw_sequence_from_video(video_path: str, task_path: str, mode: str = "video"):
    """Extract (seq[SEQ_LEN,126], present[SEQ_LEN,2]) from a single-gloss clip.

    mode="video" (default, recommended): a FRESH landmarker instance per video,
        running in VIDEO mode. Every frame is processed in order with temporal
        tracking, then SEQ_LEN frames are uniformly sampled. This matches the live
        app, which also tracks every frame and samples afterward.
    mode="image": legacy. Sample SEQ_LEN frame indices first, detect each
        independently (no tracking).

    Returns (None, None) if the clip has no frames. The instance is created and
    closed inside this call, so tracking state never leaks between videos.
    """
    cap = cv2.VideoCapture(video_path)
    frames = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    cap.release()
    if not frames:
        return None, None

    if str(mode).lower() == "video":
        landmarker = make_landmarker(task_path, mode="video")  # fresh instance per video
        per_kp, per_pres = [], []
        last_ts = 0
        for i, fr in enumerate(frames):
            ts = int(round(i * 1000.0 / max(fps, 1.0)))
            if ts <= last_ts:
                ts = last_ts + 1
            last_ts = ts
            res = detect_video(landmarker, fr, ts)   # tracked, in order
            kp, pr = landmarks_from_result(res)
            per_kp.append(kp)
            per_pres.append(pr)
        landmarker.close()
        idxs = sample_indices(len(frames))
        seq = np.stack(per_kp)[idxs].astype(np.float32)
        pres = np.stack(per_pres)[idxs].astype(np.float32)
        return seq, pres

    # legacy IMAGE mode
    landmarker = make_landmarker(task_path, mode="image")
    idxs = sample_indices(len(frames))
    seq = np.zeros((SEQ_LEN, FEATURES), dtype=np.float32)
    pres = np.zeros((SEQ_LEN, HANDS), dtype=np.float32)
    for r, ix in enumerate(idxs):
        res = detect_image(landmarker, frames[ix])
        kp, pr = landmarks_from_result(res)
        seq[r] = kp
        pres[r] = pr
    landmarker.close()
    return seq, pres


def features_from_video(video_path: str, task_path: str):
    """Full pipeline: video -> normalized (SEQ_LEN, 126). None if no frames."""
    seq, pres = raw_sequence_from_video(video_path, task_path)
    if seq is None:
        return None
    return normalize_sequence(seq, pres)

## Landmark augmentation (training set only)
Applied to the **normalized** features. These survive normalization (which already
removes global position/scale): small **rotation** (camera tilt), **jitter** (per-signer
landmark noise), **scale wobble**, **left/right mirror** (with hand-slot swap — doubles
data, covers handedness), and **time-warp** (signing-speed variation).

In [ ]:
def _slot_present(pts): return np.abs(pts).sum(axis=(2,3)) > 0
def aug_rotate(x, max_deg=15.0):
    th=np.deg2rad(np.random.uniform(-max_deg,max_deg)); c,s=np.cos(th),np.sin(th)
    p=x.reshape(-1,2,21,3).copy(); xx,yy=p[...,0].copy(),p[...,1].copy()
    p[...,0]=xx*c-yy*s; p[...,1]=xx*s+yy*c; return p.reshape(x.shape)
def aug_scale(x, lo=0.9, hi=1.1): return (x*np.random.uniform(lo,hi)).astype(np.float32)
def aug_jitter(x, sigma=0.02):
    p=x.reshape(-1,2,21,3).copy(); m=_slot_present(p)[...,None,None]
    return (p+np.random.normal(0,sigma,p.shape)*m).reshape(x.shape).astype(np.float32)
def aug_mirror(x):
    p=x.reshape(-1,2,21,3).copy(); p[...,0]*=-1.0; p=p[:,[1,0],:,:]
    return p.reshape(x.shape).astype(np.float32)
def aug_timewarp(x):
    T=x.shape[0]; a=int(np.random.uniform(0,0.2*T)); b=int(np.random.uniform(0.8*T,T-1))
    if b<=a: a,b=0,T-1
    idx=np.clip(np.round(np.linspace(a,b,T)),0,T-1).astype(int); return x[idx].astype(np.float32)
def augment(x):
    if np.random.rand()<0.5: x=aug_timewarp(x)
    if np.random.rand()<0.7: x=aug_rotate(x)
    if np.random.rand()<0.5: x=aug_scale(x)
    if np.random.rand()<0.5: x=aug_mirror(x)
    if np.random.rand()<0.7: x=aug_jitter(x)
    return x.astype(np.float32)

## Extract features (VIDEO mode, fresh instance per clip)
`features_from_video` opens a clip, runs a fresh tracked landmarker over **all** frames
in order, samples 30, and normalizes. Caches `X/y/label_map` to Drive so future runs can
skip this (set `FORCE_REEXTRACT=False`).

In [ ]:
cache = [os.path.join(OUT_DIR,f) for f in ('X.npy','y.npy','label_map.json')]
if (not FORCE_REEXTRACT) and all(os.path.exists(c) for c in cache):
    X=np.load(cache[0]); y=np.load(cache[1]); label_map=json.load(open(cache[2]))
    print('Loaded cache', X.shape)
else:
    labels = sorted(d for d in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH,d)))
    label_map = {lab:i for i,lab in enumerate(labels)}
    print('Classes:', labels)
    X, y = [], []
    for lab in labels:
        cdir = os.path.join(DATASET_PATH, lab)
        vids = [v for v in sorted(os.listdir(cdir)) if v.lower().endswith(('.mp4','.mov','.avi','.webm'))]
        for v in tqdm(vids, desc=lab):
            feat = features_from_video(os.path.join(cdir,v), TASK_PATH)  # VIDEO mode
            if feat is None: print('  skip empty:', v); continue
            X.append(feat); y.append(label_map[lab])
    X=np.array(X,np.float32); y=np.array(y)
    np.save(cache[0],X); np.save(cache[1],y); json.dump(label_map, open(cache[2],'w'))
    print('Saved', X.shape, y.shape)

labels = [k for k,_ in sorted(label_map.items(), key=lambda kv: kv[1])]
N = len(labels); T, F = X.shape[1], X.shape[2]
print(N, 'classes |', T, 'frames |', F, 'features')

## One fair split: train / val / test (test never augmented)

In [ ]:
X_tmp,X_test,y_tmp,y_test = train_test_split(X,y,test_size=TEST_SPLIT,stratify=y,random_state=SEED)
X_tr,X_val,y_tr,y_val = train_test_split(X_tmp,y_tmp,test_size=VAL_SPLIT,stratify=y_tmp,random_state=SEED)
aug_X,aug_y=[],[]
for xi,yi in zip(X_tr,y_tr):
    aug_X.append(xi); aug_y.append(yi)
    for _ in range(AUG_PER_SAMPLE): aug_X.append(augment(xi)); aug_y.append(yi)
X_tr_a=np.array(aug_X,np.float32); y_tr_a=np.array(aug_y)
perm=np.random.permutation(len(X_tr_a)); X_tr_a,y_tr_a=X_tr_a[perm],y_tr_a[perm]
cw=compute_class_weight('balanced',classes=np.arange(N),y=y_tr)
class_weight={i:float(w) for i,w in enumerate(cw)}
print('train(aug):',X_tr_a.shape,'| val:',X_val.shape,'| test:',X_test.shape)

## The three architectures

All take the same input — a `(30, 126)` sequence: 30 time-sampled frames, each a
position/scale-normalized vector of 2 hands × 21 landmarks × (x,y,z). They differ in
*how they model time*, which is the core of the comparison: **convolutional-recurrent**,
**recurrent + attention**, and **pure self-attention**.

### Model A — CNN-BiGRU (convolutional-recurrent hybrid)

**Structure:** `Conv1D(64) → Conv1D(128)` (each + BatchNorm) → `BiGRU(64, seq) → BiGRU(64)`
→ `Dense(64) → Dropout → softmax`.

**Why.** The two 1-D convolutions slide along the **time axis**, so each learns *local
motion primitives* — how the landmarks change over a handful of adjacent frames (velocity,
short sub-gestures). BatchNorm keeps those activations stable and speeds convergence. The
stacked **bidirectional GRUs** then model the sign as a sequence in *both directions*, so
the representation of any frame is informed by the whole onset–hold–offset trajectory —
important because a gloss is defined by its full movement, not a single pose. GRUs (vs
LSTMs) use fewer parameters, which suits a small dataset. This CNN-for-local +
RNN-for-global division of labour is a strong, data-efficient default and is your current
production model.

In [ ]:
from tensorflow.keras import layers, models
def build_cnn_bigru():
    inp=layers.Input((T,F))
    x=layers.Conv1D(64,3,padding='same',activation='relu')(inp); x=layers.BatchNormalization()(x)
    x=layers.Conv1D(128,3,padding='same',activation='relu')(x); x=layers.BatchNormalization()(x)
    x=layers.Bidirectional(layers.GRU(64,return_sequences=True,dropout=0.2))(x)
    x=layers.Bidirectional(layers.GRU(64,dropout=0.2))(x)
    x=layers.Dense(64,activation='relu')(x); x=layers.Dropout(0.4)(x)
    return models.Model(inp,layers.Dense(N,activation='softmax')(x),name='CNN_BiGRU')

### Model B — BiLSTM + Attention (recurrent)

**Structure:** `BiLSTM(128, seq) → BiLSTM(64, seq) → attention pooling → Dense → softmax`.

**Why.** This is the classical sequence model. **LSTM** gating (input/forget/output)
handles longer-range temporal dependencies and is the traditional baseline for gesture and
sign recognition; **bidirectional** lets it read the sign forwards and backwards. Instead
of collapsing the sequence to the last hidden state (which over-weights the end of the
sign), an **additive attention** layer learns a weight for every frame and forms a weighted
sum — so the model concentrates on the most discriminative moment (e.g. the key handshape)
and ignores the bring-in/settle frames. A practical bonus for the thesis: the attention
weights are *interpretable* — you can plot which frames the model relied on.

In [ ]:
def build_bilstm_attention():
    inp=layers.Input((T,F))
    x=layers.Bidirectional(layers.LSTM(128,return_sequences=True,dropout=0.2))(inp)
    x=layers.Bidirectional(layers.LSTM(64,return_sequences=True,dropout=0.2))(x)
    score=layers.Dense(1)(x); score=layers.Softmax(axis=1)(score)
    ctx=layers.Multiply()([x,score]); ctx=layers.Lambda(lambda z: tf.reduce_sum(z,axis=1))(ctx)
    d=layers.Dense(64,activation='relu')(ctx); d=layers.Dropout(0.4)(d)
    return models.Model(inp,layers.Dense(N,activation='softmax')(d),name='BiLSTM_Attention')

### Model C — Transformer encoder (self-attention)

**Structure:** `Dense(d_model) + positional embedding → ×2 [MultiHeadAttention → FFN]
with residual+LayerNorm → GlobalAveragePooling → Dense → softmax`.

**Why.** The modern attention paradigm, included so the comparison spans all three model
families. The landmark vector is projected to `d_model` and, because self-attention is
order-agnostic, **learnable positional embeddings** inject frame order. **Multi-head
self-attention** then lets every frame attend directly to every other frame in one step —
capturing long-range relations (e.g. start pose vs end pose) without the sequential
bottleneck of an RNN, and in parallel. Multiple heads attend to different relation types.
Transformers are data-hungry, so a useful thesis discussion point is whether augmentation
lets it match the recurrent/hybrid models at this dataset size.

In [ ]:
class PositionalEmbedding(layers.Layer):
    def __init__(self, T, d, **kw):
        super().__init__(**kw); self.T=T; self.pos=layers.Embedding(T,d)
    def call(self,x): return x + self.pos(tf.range(self.T))

def build_transformer(d_model=128, heads=4, ff=256, blocks=2, drop=0.2):
    inp=layers.Input((T,F)); x=layers.Dense(d_model)(inp); x=PositionalEmbedding(T,d_model)(x)
    for _ in range(blocks):
        a=layers.MultiHeadAttention(num_heads=heads,key_dim=d_model//heads,dropout=drop)(x,x)
        x=layers.LayerNormalization(epsilon=1e-6)(x+a)
        f=layers.Dense(ff,activation='relu')(x); f=layers.Dense(d_model)(f); f=layers.Dropout(drop)(f)
        x=layers.LayerNormalization(epsilon=1e-6)(x+f)
    x=layers.GlobalAveragePooling1D()(x); x=layers.Dense(64,activation='relu')(x); x=layers.Dropout(0.4)(x)
    return models.Model(inp,layers.Dense(N,activation='softmax')(x),name='Transformer')

BUILDERS={'CNN_BiGRU':build_cnn_bigru,'BiLSTM_Attention':build_bilstm_attention,'Transformer':build_transformer}

## Metric & figure helpers (saved to Drive)

In [ ]:
def save_history_plot(hist,name):
    h=hist.history; fig,ax=plt.subplots(1,2,figsize=(11,4))
    ax[0].plot(h['accuracy'],label='train'); ax[0].plot(h['val_accuracy'],label='val')
    ax[0].set_title(f'{name} — Accuracy'); ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(h['loss'],label='train'); ax[1].plot(h['val_loss'],label='val')
    ax[1].set_title(f'{name} — Loss'); ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.savefig(os.path.join(METRICS_DIR,f'{name}_curves.png'),bbox_inches='tight'); plt.show()

def save_confusion(yt,yp,name,normalize=False):
    cm=confusion_matrix(yt,yp,labels=np.arange(N)).astype(float)
    if normalize: cm=cm/np.clip(cm.sum(1,keepdims=True),1,None)
    fig,ax=plt.subplots(figsize=(min(1+0.5*N,14),min(1+0.5*N,14)))
    im=ax.imshow(cm,cmap='Blues'); fig.colorbar(im,fraction=0.046)
    ax.set_xticks(range(N)); ax.set_xticklabels(labels,rotation=90,fontsize=8)
    ax.set_yticks(range(N)); ax.set_yticklabels(labels,fontsize=8)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'{name} — Confusion'+(' (normalized)' if normalize else ' (counts)'))
    thr=cm.max()/2 if cm.max()>0 else .5
    for i in range(N):
        for j in range(N):
            v=cm[i,j]
            if v>0: ax.text(j,i,(f'{v:.2f}' if normalize else f'{int(v)}'),ha='center',va='center',fontsize=7,color='white' if v>thr else 'black')
    plt.tight_layout(); plt.savefig(os.path.join(METRICS_DIR,f'{name}_{"cm_norm" if normalize else "cm"}.png'),bbox_inches='tight'); plt.show()

def save_perclass_f1(rep,name):
    f1=[rep[l]['f1-score'] for l in labels]
    plt.figure(figsize=(min(1+0.5*N,14),4)); plt.bar(range(N),f1)
    plt.xticks(range(N),labels,rotation=90,fontsize=8); plt.ylim(0,1); plt.ylabel('F1')
    plt.title(f'{name} — per-class F1'); plt.grid(axis='y',alpha=.3)
    plt.tight_layout(); plt.savefig(os.path.join(METRICS_DIR,f'{name}_f1.png'),bbox_inches='tight'); plt.show()

## Train + evaluate all three (metrics on the held-out test set)

In [ ]:
results,histories={},{}; y_test_oh=np.eye(N)[y_test]
for name,builder in BUILDERS.items():
    print('\n'+'='*60+f'\nTraining {name}\n'+'='*60)
    tf.keras.backend.clear_session(); np.random.seed(SEED); tf.random.set_seed(SEED)
    model=builder(); model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                   loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    cbs=[tf.keras.callbacks.EarlyStopping(monitor='val_accuracy',patience=15,restore_best_weights=True),
         tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',factor=0.5,patience=6,min_lr=1e-5)]
    t0=time.time(); hist=model.fit(X_tr_a,y_tr_a,validation_data=(X_val,y_val),epochs=EPOCHS,
        batch_size=BATCH_SIZE,class_weight=class_weight,callbacks=cbs,verbose=2); train_time=time.time()-t0
    histories[name]=hist
    probs=model.predict(X_test,verbose=0); y_pred=probs.argmax(1)
    _=model.predict(X_test[:1],verbose=0); ti=time.time(); _=model.predict(X_test,verbose=0)
    infer_ms=(time.time()-ti)/len(X_test)*1000
    rep=classification_report(y_test,y_pred,target_names=labels,output_dict=True,zero_division=0)
    pd.DataFrame(rep).transpose().to_csv(os.path.join(METRICS_DIR,f'{name}_report.csv'))
    try: auc=roc_auc_score(y_test_oh,probs,multi_class='ovr',average='macro')
    except Exception: auc=float('nan')
    mfile=os.path.join(METRICS_DIR,f'{name}.keras'); model.save(mfile)
    results[name]={'params':int(model.count_params()),'size_MB':round(os.path.getsize(mfile)/1e6,2),
        'train_time_s':round(train_time,1),'infer_ms':round(infer_ms,2),
        'test_accuracy':round(accuracy_score(y_test,y_pred),4),
        'top3_accuracy':round(top_k_accuracy_score(y_test,probs,k=3,labels=np.arange(N)),4),
        'macro_F1':round(rep['macro avg']['f1-score'],4),'weighted_F1':round(rep['weighted avg']['f1-score'],4),
        'cohen_kappa':round(cohen_kappa_score(y_test,y_pred),4),'mcc':round(matthews_corrcoef(y_test,y_pred),4),
        'macro_AUC':round(float(auc),4),'best_val_acc':round(max(hist.history['val_accuracy']),4)}
    print(json.dumps(results[name],indent=2))
    save_history_plot(hist,name); save_confusion(y_test,y_pred,name,False)
    save_confusion(y_test,y_pred,name,True); save_perclass_f1(rep,name)

## Cross-model comparison

In [ ]:
summary=pd.DataFrame(results).transpose()
summary.to_csv(os.path.join(METRICS_DIR,'comparison_summary.csv'))
json.dump(results,open(os.path.join(METRICS_DIR,'comparison_summary.json'),'w'),indent=2)
print(summary.to_string())

fig,ax=plt.subplots(1,2,figsize=(12,4))
for name,h in histories.items():
    ax[0].plot(h.history['val_accuracy'],label=name); ax[1].plot(h.history['val_loss'],label=name)
ax[0].set_title('Validation accuracy'); ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].set_title('Validation loss'); ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(METRICS_DIR,'compare_curves.png'),bbox_inches='tight'); plt.show()

names=list(results.keys()); acc=[results[n]['test_accuracy'] for n in names]; f1=[results[n]['macro_F1'] for n in names]
xx=np.arange(len(names)); w=0.38; plt.figure(figsize=(7,4))
plt.bar(xx-w/2,acc,w,label='Test accuracy'); plt.bar(xx+w/2,f1,w,label='Macro F1')
plt.xticks(xx,names,rotation=15); plt.ylim(0,1); plt.grid(axis='y',alpha=.3); plt.legend()
for i,(a,b) in enumerate(zip(acc,f1)): plt.text(i-w/2,a+.01,f'{a:.2f}',ha='center',fontsize=8); plt.text(i+w/2,b+.01,f'{b:.2f}',ha='center',fontsize=8)
plt.title('Model comparison (held-out test)'); plt.tight_layout()
plt.savefig(os.path.join(METRICS_DIR,'compare_bars.png'),bbox_inches='tight'); plt.show()
print('\nSaved to:', METRICS_DIR)

## Deploy the best model
Pick the winner from the table, then copy its files into the project root:
`thesis_metrics/<MODEL>.keras` → `model.keras`, and the `label_map.json` from `HAND_MODEL/`.
Set `config.json` → `feature_normalize: "sequence"`. All three are drop-in compatible
because the app shares this notebook's `sign_features` preprocessing.